In [1]:
!pip install ddgs trafilatura -q
!pip install openai-agents

  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached cffi-2.0.0-cp314-cp314-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ---------------------------------------- 0.0/816.1 kB ? eta -:--:--
   ---------------------------------------- 816.1/816.1 kB 11.8 MB/s  0:00:00
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 3.8/3.8 MB 35.9 MB/s  0:00:00
Using cached cffi-2.0.0-cp314-cp314-win_amd64.whl (185 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 9.7/9.7 MB 65.8 MB/s  0:00:00
Using cached pycparser-3.0-py3-none-any.whl (48 kB)

   ----------------------------------------  0/11 [pywin32]
   ----------------------------------------  0/11 [pywin32]
   ----------------------------------------  0/11 [pywin32]
   ---------------

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from openai import OpenAI
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura
from pprint import pprint
from IPython.display import Markdown, display

from agents import Agent, Runner, function_tool



load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
    raise Exception("API Key is missing!")
else:
    print("Key is: " + OPENAI_API_KEY[:8])

client = OpenAI(api_key=OPENAI_API_KEY)

Key is: sk-proj-


In [42]:
@function_tool
def search_web(query:str) -> str:
    ddgs = DDGS()

    results = ddgs.text(query, max_result = 5)
    #pprint(f"Got results: {results}")
    return json.dumps(results, indent=2)

In [43]:
@function_tool
def fetch_url(url: str) -> str:

    downloaded = trafilatura.fetch_url(url)

    if downloaded:
        text = trafilatura.extract(downloaded)
        print(text)
        if text: 
            print(f" \u2705 Got Text")
            return text
    print(f"\u274C fail to fetch text from URL {url}. ")
    return (f"Could not extract text from url {url}. Try different source.")


In [ ]:
JUDGE_MODEL = "gpt-4.1"
MODEL = "gpt-4.1-nano"

## The Agents

The agent prompt tells LLM who it is and how to behave.
The key things:
- What its job is
- What tools it has
- What process to follw
- What output format to produce

#### ResearchAgent


In [ ]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief
 
Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""



research_agent = Agent(
    name="ResearchAgent",
    instructions=RESEARCH_AGENT_PROMPT,
    model = MODEL,
    tools = [search_web, fetch_url]
)


#### Orchestrator Agent



In [ ]:
ORCHESTRATOR_AGENT_PROMPT = """You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs 
to get 2 research briefs.
You pick the best research brief out of the two and deliver it as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.
"""

orchestrator_agent = Agent(
    model="o4-mini",
    name="Orchestrator Agent",
    instructions = ORCHESTRATOR_AGENT_PROMPT,
    tools = [research_agent.as_tool(
                tool_name="reseearch agent",
                tool_description="Research topic on the internet, return brief with key facts, themes, statistic",


                    )]
)

In [48]:
result = await Runner.run(
    research_agent,
    input="Research the following topic and produce a comprehensive brief on Ai in Healthcare in 2030",
    max_turns=30
)

In [49]:
print(f"Agent: {result.last_agent.name}")
print("------")
display(Markdown(result.final_output))

Agent: ResearchAgent
------


Based on recent web searches and available literature, here is a comprehensive research brief on the future of AI in healthcare by 2030:

### Key Facts and Statistics:

- The global AI healthcare market is projected to reach approximately **$187.95 billion** by 2030, with a compound annual growth rate (CAGR) of about **37%** from 2023 (Ref: Voiceoc).
- AI's potential economic impact in healthcare is immense, with the industry expected to generate around **US$ 868 billion** through innovations like precision medicine, AI-driven trial management, and consumer health platforms (Ref: Strategy&).
- An estimated **4.5 billion people** worldwide currently lack access to essential healthcare services, highlighting AI’s potential to bridge gaps in healthcare access and address a projected **11 million health worker shortage** by 2030 (Ref: World Economic Forum).

### Main Themes and Arguments:

1. **Transforming Diagnostics and Treatment:**
   AI is set to revolutionize diagnostics through advanced imaging, predictive analytics, and personalized medicine. AI-driven tools will support earlier disease detection, more accurate diagnoses, and tailored treatment plans, improving outcomes and efficiency (Refs: Mayo Clinic, Nature Medicine).

2. **Enhancing Healthcare Delivery and Workflow:**
   Automation of administrative tasks, clinical documentation via large language models (LLMs), and decision support systems will alleviate clinician burnout and streamline healthcare workflows, resulting in faster and more effective patient care (Refs: PMC, World Economic Forum).

3. **Global Health Equity and Accessibility:**
   AI has the potential to address disparities by providing accessible diagnostics and treatment options, especially in underserved regions. It could play a pivotal role in delivering healthcare in remote areas, thus improving global health equity (Refs: WHO, Strategy&).

4. **Innovation in Drug Discovery and Precision Medicine:**
   AI will accelerate drug discovery, reduce costs, and enhance precision medicine. Machine learning algorithms will facilitate faster clinical trials, identification of therapeutic targets, and personalized treatment options (Ref: Deloitte, McKinsey).

5. **Predictive and Preventive Healthcare:**
   Predictive analytics will forecast health events and surges in demand, enabling proactive interventions and resource allocation. Real-time monitoring via wearable devices and AI-powered diagnostics will become commonplace (Refs: Harvard Medical School, PwC).

6. **Ethical, Regulatory, and Data Security Challenges:**
   Despite significant advancements, integrating AI into healthcare raises ethical issues including bias, privacy, and accountability. Addressing these challenges will be crucial for responsible implementation (Refs: PMC, Nature Medicine).

7. **Emergence of AI Agents and Collaborations:**
   Future healthcare will involve AI agents capable of analyzing complex, distributed data ecosystems, fostering collaborative and proactive care models that enhance patient outcomes and operational efficiency (Ref: HIMSS, PMC).

### Notable Data Points:

- CAGR of 38.5% predicted for the AI healthcare market from 2024-2030.
- AI will help solve health worker shortages by making healthcare delivery more efficient and accessible.
- The industry’s value is expected to approach nearly **$188 billion** by 2030.

### Sources:
- Mayo Clinic Press: [AI in Healthcare](https://mcpress.mayoclinic.org/healthy-aging/ai-in-healthcare-the-future-of-patient-care-and-health-management/)
- Strategy&: [AI Healthcare Revolution](https://www.strategyand.pwc.com/de/en/industries/pharma-life-sciences/ai-healthcare-revolution.html)
- Voiceoc: [AI Healthcare Trends](https://www.voiceoc.com/blogs/ai-healthcare-trends-for-future)
- World Economic Forum: [AI Transforming Global Health](https://www.weforum.org/stories/2025/08/ai-transforming-global-health/)
- Nature Medicine: [AI Improving Healthcare](https://www.nature.com/articles/s41591-026-04329-2)

### Summary:
By 2030, AI is expected to dramatically reshape healthcare, making it more personalized, efficient, and accessible. The technology will play a central role in diagnostics, treatment, drug discovery, and operational workflows while also presenting challenges that require careful ethical and regulatory governance. The industry’s growth will be driven by technological innovations, an imperative to address global health disparities, and the need for more sustainable healthcare systems.

Would you like a more detailed exploration of specific themes, such as ethical considerations or technological innovations?